In [0]:
# Connect to ADLS
storage_account_name = "azureetlstorage"

storage_account_key = dbutils.secrets.get(
    scope = "etl-scope",
    key   = "adls-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("connected!")

In [0]:
# Read reviews JSON from landing
reviews_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/landing/reviews.json"

reviews_df = spark.read.format("json") \
    .option("multiLine", "true") \
    .load(reviews_path)

print(f"Total reviews: {reviews_df.count()}")
reviews_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Add metadata columns
reviews_bronze_df = reviews_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id",    F.lit("batch_001"))

# Write to Bronze
reviews_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/reviews"

reviews_bronze_df.write.format("delta") \
    .mode("overwrite") \
    .save(reviews_bronze_path)

print(f"Bronze reviews written — {reviews_bronze_df.count()} rows")